In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install faiss-cpu 
import warnings
warnings.filterwarnings("ignore")

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 
print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)
print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.2 MB/s eta 0:00:00
Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
!pip install transformers
from transformers import pipeline
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150['answer'])
opt=str(row_150[ans_150])

result = zs(prompt_150, candidate_labels=labels_150)
# correct_opt = str(row_150[ans_150]) 
score = None
for label, prob in zip(result['labels'], result['scores']):
    if label == opt:
        score = prob
print(score)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

0.38442012667655945


In [4]:
prompt_text = str(train.iloc[150]["prompt"])
query_embd = model.encode([prompt_text], show_progress_bar=False)
k = 10
distances, indices = index.search(query_embd, k)
print(indices[0])
index_ = 150
rank = None
for i, idx in enumerate(indices[0], start=1):
    if idx == index_:
        rank = i
        break
print(rank)

[ 663 1701 1269 1532  576  847 1693 1906  168  150]
10


In [5]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
retrieved_indices = indices[0] 
docs_10 = [kb[i] for i in retrieved_indices]
prompt_150 = str(train.iloc[150]["prompt"])
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)
results = sorted(zip(docs_10, retrieved_indices, ce_scores), key=lambda x: x[2], reverse=True)
rank = None
for i, (_, idx, _) in enumerate(results, start=1):
    if idx == index_:
        rank = i
        break
print(rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


In [6]:
from transformers import AutoTokenizer
prompt_42 = str(train.iloc[42]["prompt"])
query_embd=model.encode([prompt_42], show_progress_bar=False)
k = 5
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)
distances,indices = index.search(query_embd, k)
retrieved_docs = [kb[i] for i in indices[0]]
concatenated_docs = " ".join(retrieved_docs)
context_string = f"Context: {concatenated_docs} Question: {prompt_42}"
tokenizer =AutoTokenizer.from_pretrained("bert-base-uncased")
tokens =tokenizer(context_string, truncation=False)
total_tokens = len(tokens["input_ids"])
print(total_tokens)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [7]:
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
ans = str(row_150["answer"])
correct_opt = str(row_150[ans])
true_doc = kb[150]
rag_string = f"Context: {true_doc} Question: {prompt_150}"
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),str(row_150['D']), str(row_150['E'])]
result = zs(rag_string, candidate_labels=labels_150)
score = None
for label, prob in zip(result['labels'], result['scores']):
    if label == correct_opt:
        score = prob
        break
print(score)

0.9894258379936218


In [8]:
adversarial_doc = kb[999]
adversarial_rag = f"Context: {adversarial_doc} Question: {prompt_150}"
result = zs(adversarial_rag, candidate_labels=labels_150)
score = None
for label, prob in zip(result['labels'], result['scores']):
    if label == correct_opt:
        score = prob
        break
print(score)

0.5289487838745117


In [9]:
hits = 0
total = 100
k = 5
for i in range(total):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    ans = str(row["answer"])
    correct_opt = str(row[ans])
    query_embd = model.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(query_embd, k)
    retrieved_docs = [kb[j] for j in indices[0]]
    if any(correct_opt in doc for doc in retrieved_docs):
        hits +=1
hit_rate = (hits/total)*100
print(hit_rate)

73.0


In [10]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
def mapat3(correct_letter,ranked_letters):
    score = 0.0
    for i, letter in enumerate(ranked_letters[:3], start=1):
        if letter == correct_letter:
            score = 1.0 / i
            break
    return score
scores = []
k = 5
for i in range(20):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    ans = str(row["answer"])
    options = [str(row['A']), str(row['B']), str(row['C']),str(row['D']), str(row['E'])]
    query_embd = model.encode([prompt], show_progress_bar=False)
    _, indices = index.search(query_embd, k)
    docs = [kb[j] for j in indices[0]]
    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs[int(ce_scores.argmax())]
    rag_string = f"Context: {best_doc} Question: {prompt}"
    result = zs(rag_string, candidate_labels=options)
    ranked_labels = [label for label, _ in sorted(zip(result['labels'], result['scores']),key=lambda x: x[1],reverse=True)]
    letter = {str(row['A']): 'A',str(row['B']): 'B',str(row['C']): 'C',str(row['D']): 'D',str(row['E']): 'E'}
    ranked_letters = [letter[l] for l in ranked_labels]
    score = mapat3(ans, ranked_letters)
    scores.append(score)
avg_map3 = sum(scores) / len(scores)
print(avg_map3)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
